# 04 记账桌面APP开发

## 1. 本节目标与资料来源

本篇对应《VibeCoding实战》的 **09—入门项目：黑马记账桌面APP开发**。在已有 Node.js、Git、Python、Claude Code 和 VSCode 环境上，开发一个能在 Windows、macOS 运行的个人记账桌面应用。

课程以初学者的视角推进：描述产品需求，让 Claude Code 列出技术方案、解释优劣，再由用户选择。开发过程包括产品文档、项目初始化、功能实现、错误修复和双平台打包。

资料来自课程配套讲义 [入门实战项目](https://www.yuque.com/xxcls/vibecoding/wmmswml3wi874b7v) 和 [课程网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)（提取码 `1234`）。本篇已读取完整讲义、两段原始提示词及其教学截图，保留关键原图；没有逐段转录视频，也没有读取记账项目的全部源码。

以下按讲义顺序整理。标为“补充”的内容用于解释当前步骤；课程中 AI 给出的体积、速度和费用，只代表当时的方案估计或演示结果。

## 2. 创建项目文件夹

在本地创建文件夹 **黑马记账 APP**，用 VSCode 的“打开文件夹”打开它，再进入 Claude Code 插件。讲义截图中的目录也写作 `黑马记账APP`，有无空格不影响产品名称，命令中的路径必须与实际目录一致。

首次打开自己创建的项目时，VSCode 可能询问是否信任作者。课程确认信任后继续开发。此时资源管理器中应是空项目，而不是其他已有工程。

本节沿用上一篇的基础环境；选定 Tauri 之后，还会检查并补齐 Rust 和平台编译工具。

> 讲义中的命令多在 macOS 的 Bash 环境执行。下文省略作者的个人绝对路径和用于合并日志的 `2>&1`；每段命令前说明工作目录。Windows PowerShell 的路径及环境加载方式另行标明。

## 3. 选择 Claude Code 的工作模式

课程先打开输入框附近的模式菜单，选择 **Plan mode**，并把 **Effort** 调到较高档位，让 Claude 先解释方案。

<img src="images/04-01-claude-modes.png" alt="课程中的 Claude Code 模式菜单" width="480">

| 课程界面选项 | 讲义中的作用 |
|---|---|
| `Ask before edits` | 修改文件前请求用户批准 |
| `Edit automatically` | 自动执行文件编辑；讲义所用配置中，Shell 命令仍需批准 |
| `Plan mode` | 先了解项目、提出计划，确认后再进入实施 |
| `Effort (max)` | 调整思考投入，课程用较高投入来讨论方案 |

这些名称与审批行为对应课程当时的插件版本和权限配置。较高 Effort 通常增加时间或消耗，不能据此保证结果正确。

本课的协作原则是：**AI 提出并解释技术选择，用户决定产品取舍。** 看不懂某个命令时，先要求解释作用和影响，再批准该步骤。

## 4. 用原始提示词生成产品文档

**PRD** 是产品需求文档，用来明确应用解决什么问题、提供哪些功能。课程同时把产品要求和 AI 协作规则写入项目文档，便于后续开发持续参考。

讲义中的原始提示词：

```text
我需要生成一个APP，名字：黑马记账；我不懂编程技术，也是初次使用claude code，你先帮我设计产品文档，并记录到claude.md中；
我的一些要求如下：
1. 我需要这个APP是可以运行在Windows电脑和Mac电脑上；
2. APP功能主要是可以记录用户每一次的花销（单位人民币），并可以记录花销的分类，分类分为2级，有一级大类、和二级小类，分类你来设计；
3. 技术栈选型你决定，请列出几个方案给我并列出优劣势，我来挑选并决定；

注意：在整个项目开发过程中，任何关于技术方面的事情，我无法提供具体的要求，必须是你列出方案给我，并向我解释各个方案，由我来决定，请将这个要求详细记录在claude.md中，在整个项目期间遵守；
```

这里先给出平台、功能、币种和分类层级，并要求 AI 解释技术选择。用户不用先决定数据库、UI 框架或打包工具。

**文件名说明：**提示词和讲义部分文字使用 `claude.md`，实际写入截图中的文件名为 **`CLAUDE.md`**。本篇后续按实际文件名记录，避免在大小写敏感的系统上当成两个文件。

`CLAUDE.md` 在本课承担两项职责：保存产品需求，以及记录“先解释、列方案、由用户决定”的项目协作约定。它提供上下文，不能替代构建检查和功能验证。

## 5. 确认桌面方案、分类与功能

Claude Code 阅读空目录后提出三种桌面技术方案，课程选择 **方案 B：Tauri + React**。

<img src="images/04-02-desktop-options.png" alt="课程提出的三种桌面技术方案" width="760">

| 方案 | 课程讨论的主要取舍 | 本课选择 |
|---|---|---|
| A：Electron + React | 桌面生态成熟，通常需要更大的安装包和运行内存 | 未选 |
| B：Tauri + React | 使用系统 WebView 展示界面，较轻量；构建需要 Rust 与平台工具链 | 选用 |
| C：Flutter Desktop | 可考虑后续扩展到移动端，采用另一套 UI 技术 | 未选 |

截图中的约 `10 MB`、`150 MB` 等数值是当时 AI 的方案估计，不能作为所有项目的固定体积。

随后确认两组产品选择：

1. **分类不调整**：接受 AI 设计的 `10` 个一级分类、`60+` 个二级分类。
2. **增加功能**：勾选“月度收支统计图表”和“数据导出 Excel/CSV”，再点击 `Submit answers`。

<img src="images/04-03-feature-selection.png" alt="课程勾选月度统计与数据导出" width="560">

“预算设置与提醒”出现在备选项中，课程此处没有勾选。讲义正文的“阅读支出统计图表”是文字笔误，截图显示的是“月度收支统计图表”。最终演示主要围绕**支出**记录与统计展开，不能仅凭选项名称推断已实现收入管理。

## 6. 批准计划并写入 CLAUDE.md

Claude 给出最终计划后，课程选择 **`2 Yes, and manually approve edits`**：接受方案，但后续文件编辑仍逐次确认。

生成 `CLAUDE.md` 时，插件会显示 `Allow write to CLAUDE.md?`。确认内容后选择 `Yes`，资源管理器中应出现实际文件。

`Yes, allow all edits this session` 表示本会话后续文件编辑自动批准。它与当前这一次的 `Yes` 范围不同；课程后面为了减少重复操作也展示了这一选项。

已确定的 V1 工作顺序：

1. 创建 `CLAUDE.md`，写入产品文档与技术决策规则。
2. 初始化 Tauri + React + TypeScript 项目。
3. 决定数据存储方式，组织分类与账单数据。
4. 实现金额、分类、日期、备注输入，以及账单列表。
5. 实现月度统计图表。
6. 实现 Excel/CSV 导出。
7. 测试并打包 macOS 和 Windows 应用。

**此时数据库方案尚未最终决定。** 初始规划中出现数据库设计与 SQLite 的讨论，后续实际选择的是 IndexedDB + Dexie，不能把计划阶段的候选项写成已经实现的方案。

## 7. 检查 Node.js、npm 与 Rust 环境

课程进入初始化前，先检查工具版本。以下命令可在终端分别执行：

```bash
node --version
npm --version
rustc --version
cargo --version
```

| 工具 | 本项目中的职责 |
|---|---|
| Node.js、npm | 安装前端依赖，运行 Vite 和 Tauri CLI |
| `rustc` | 编译 Tauri 的 Rust 代码 |
| `cargo` | 管理 Rust 依赖并执行构建 |
| `rustup` | 安装和管理 Rust 工具链 |

演示中 Node.js、npm 已就绪，Rust 尚未安装。课程在 **macOS** 使用 rustup 安装：

```bash
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
source "$HOME/.cargo/env"
```

第一条获取并运行 rustup 安装脚本，`-y` 使用默认安装选项；第二条让当前 Shell 加载 Cargo 的环境。安装完成后重新检查 `rustc`、`cargo`。

**补充：**以上 `source` 和管道安装写法属于 macOS/Linux Shell，不是 PowerShell 命令。Windows 的 Rust 安装和 MSVC 编译工具在后文单独记录。

## 8. 初始化失败后，改为手动创建项目

Rust 安装完成后，Claude 首先尝试 Tauri 脚手架。在**黑马记账项目根目录**执行的核心命令是：

```bash
npm create tauri-app@latest . -- --template react-ts --manager npm
```

`.` 表示当前目录，`react-ts` 指 React + TypeScript 模板，`--manager npm` 指定包管理器。

演示中交互式命令不可用，Claude 改尝试 Vite：

```bash
npm create vite@latest . -- --template react-ts
```

<img src="images/04-04-scaffold-fallback.png" alt="脚手架交互失败后改用 Vite 的课程记录" width="760">

随后又尝试了管道输入，最后转为**逐个写入项目文件**。因此本课的实际过程并非“一条脚手架命令成功生成所有内容”。上面是先后尝试的命令，不是要求在同一目录把两套脚手架都重新执行一遍。

### npm 包名与应用显示名

写 `package.json` 时，AI 使用 `heimajizhang`。课程追问：如果项目名允许中文就使用“黑马记账”，否则保持拼音。

最终保留：

- npm 包名：`heimajizhang`，供包管理和项目配置使用。
- 应用显示名称：`黑马记账`，供窗口与桌面产品展示。

两者用途不同，不需要把 `package.json` 的 `name` 强行改成中文。

## 9. 查看生成的前端与 Tauri 文件

手动搭建后，课程先生成 React 前端文件，再创建 `src-tauri/src`、`src-tauri/icons`、`src-tauri/capabilities`，写入 Rust 入口、配置与权限文件。

下面根据课程文件树和编辑截图汇总关键文件，省略其他文件：

```text
黑马记账APP/
├── CLAUDE.md
├── package.json
├── index.html
├── tsconfig.json
├── vite.config.ts
├── src/
│   ├── App.tsx
│   ├── main.tsx
│   ├── index.css
│   └── vite-env.d.ts
└── src-tauri/
    ├── Cargo.toml
    ├── tauri.conf.json
    ├── src/
    │   ├── main.rs
    │   └── lib.rs
    ├── icons/
    └── capabilities/
        └── default.json
```

<img src="images/04-05-frontend-files.png" alt="课程生成的前端文件树" width="420">

| 文件或目录 | 作用 |
|---|---|
| `src/` | React 界面、交互和前端业务代码 |
| `package.json` | JavaScript 依赖与 npm 脚本 |
| `vite.config.ts` | 开发服务器和前端构建配置 |
| `src-tauri/src/` | Tauri 的 Rust 桌面入口 |
| `Cargo.toml` | Rust 项目信息与依赖 |
| `tauri.conf.json` | 应用名称、窗口、前端路径和打包配置 |
| `capabilities/default.json` | Tauri 能力权限配置 |

课程 `package.json` 中的脚本片段：

```json
{
  "scripts": {
    "dev": "vite",
    "build": "tsc -b && vite build",
    "preview": "vite preview",
    "tauri": "tauri"
  }
}
```

这里是相关字段摘录，不能直接用来覆盖完整 `package.json`。`build` 先执行 TypeScript 检查，再执行 Vite 构建；前一步失败时，`&&` 后面的命令不会执行。

## 10. 接好前端构建与桌面配置

课程在 `src-tauri/tauri.conf.json` 中设置了这些关键字段：

| 字段 | 截图中的值 | 说明 |
|---|---|---|
| `productName` | `黑马记账` | 产品显示名称 |
| `identifier` | `com.heimajizhang.app` | 应用标识 |
| `build.frontendDist` | `../dist` | 前端构建产物路径，相对于 Tauri 配置所在目录 |
| `build.devUrl` | `http://localhost:1420` | 开发时连接的前端地址 |
| `build.beforeDevCommand` | `npm run dev` | 启动桌面开发前运行前端服务 |
| `build.beforeBuildCommand` | `npm run build` | 正式打包前构建前端 |
| `app.windows[0].title` | `黑马记账` | 第一个窗口的标题 |
| 窗口 `width`、`height` | `480`、`800` | 初始窗口尺寸 |

Vite 配置把开发端口设为 `1420`，与 `devUrl` 对应。端口和前端输出路径必须衔接，否则可能出现开发窗口打不开页面或打包找不到前端文件的问题。

### 加入 Tailwind CSS

课程随后在 `package.json` 加入 `@tailwindcss/vite` 和 `tailwindcss`，并修改 `vite.config.ts`：

```typescript
import tailwindcss from "@tailwindcss/vite";
```

原来的 `plugins: [react()]` 改为：

```typescript
plugins: [react(), tailwindcss()],
```

<img src="images/04-06-tailwind-vite.png" alt="课程为 Vite 加入 Tailwind 插件" width="760">

这是现有配置中的导入和字段修改，不能把两个片段当成完整文件。初始规划里出现过 `tailwind.config.js`，实际演示展示的是 Tailwind CSS 4 的 Vite 插件接入，不需要仅凭规划树再补一份旧式配置。

## 11. 安装依赖，先检查前端能否构建

在包含 `package.json` 的**项目根目录**，课程准备执行：

```bash
npm install
```

用户先追问：

```text
这个命令是做什么的？你先解释一下，然后我再决定 Yes or No。
```

这是对课程截图中追问的文字整理。Claude 解释后，用户发送“允许执行”。

<img src="images/04-07-explain-npm-install.png" alt="课程要求 Claude 解释 npm install 的作用" width="760">

`npm install` 按项目依赖声明下载所需包，并在本地形成 `node_modules`；`package-lock.json` 记录解析出的版本。这里安装的是 React、Tauri API、Vite、TypeScript、样式工具等项目依赖，不是在安装最终的记账桌面应用。

依赖安装完成后，课程执行：

```bash
npx vite build
```

成功时应看到 Vite 构建完成并产生 `dist/`。这一阶段先确认前端代码能够打包，之后再推进业务功能。

**补充：**`npx vite build` 与 `npm run build` 有区别。本课脚本中的 `npm run build` 还会执行 `tsc -b`，不能把 Vite 构建成功视为 TypeScript 检查也已通过，更不能视为桌面打包和所有功能都已验证。

## 12. 选择数据存储：IndexedDB + Dexie

继续实现账单前，Claude 按 `CLAUDE.md` 中的约定再次提出技术选择。课程选择 **方案 A**。

| 方案 | 数据在哪里 | 课程讨论的取舍 |
|---|---|---|
| A：IndexedDB + Dexie.js | 应用 WebView 的本地数据库 | 业务主要在前端实现，开发较直接 |
| B：Tauri 后端 + SQLite | 本地 SQLite 数据库文件 | 需要 Rust 与前端通信，便于以数据库文件组织和迁移数据 |

<img src="images/04-08-storage-options.png" alt="课程的数据存储方案比较" width="760">

**IndexedDB** 是浏览器环境提供的本地数据库；**Dexie** 是简化 IndexedDB 操作的 JavaScript 库。Tauri 使用 WebView 展示前端，所以本项目可以直接在前端保存分类和账单，不必为了记账功能另外实现 Rust 数据库接口。

课程选择理由是个人记账数据量较小，希望先完成可用版本。选择后，在项目根目录安装：

```bash
npm install dexie
```

后续截图中的 `../data/db` 和 `db.records` 表明，账单访问通过前端数据库模块完成。本篇没有完整数据库源码，因此不补造表结构、索引、字段类型或升级迁移代码。

**补充：**截图将风险概括为“清理缓存可能丢数据”。更准确地说，清除对应的应用或站点存储可能删除 IndexedDB 数据；普通资源缓存和数据库并非同一项。导出功能提供数据副本，不能据此认为已具备完整的数据库备份与恢复机制。

## 13. 实现记账与账单列表，再次构建

Claude 根据已确认的方案继续编写功能。课程任务列表中的核心操作包括：

- **记一笔**：金额、一级与二级分类、日期、可选备注。
- **账单列表**：查看记录，按时间倒序组织。
- **月度查看**：选择月份，显示当月支出并为统计提供数据。

AI 完成核心文件后，课程再次在项目根目录执行：

```bash
npx vite build
```

随后启动前端预览，截图中的命令是：

```bash
npx vite --host 0.0.0.0
```

这里的“开发服务器”用于预览网页界面，不是最终桌面 EXE。`--host 0.0.0.0` 让服务监听所有网络接口；**补充：**仅在本机预览时，可以按项目脚本使用 `npm run dev`，浏览器打开终端显示的本地地址。

这一轮先检查页面能否展示、录入与列表是否衔接，再继续添加图表和导出。开发过程中每次批准文件修改或命令时，都应结合它正在完成的任务看待；讲义里重复出现的“Yes”不等于独立的实现步骤。

## 14. 选择统计图表库：Recharts

实现月度统计时，Claude 提出两种图表方案：

| 方案 | 用法与取舍 |
|---|---|
| A：Recharts | 以 React 组件使用，饼图、柱状图较容易接入当前项目 |
| B：Chart.js + react-chartjs-2 | 图表生态成熟，通过 React 包装层接入，需要处理相应配置 |

<img src="images/04-09-chart-options.png" alt="课程中的 Recharts 与 Chart.js 方案比较" width="680">

课程选 **A：Recharts**，主要考虑与现有 React 技术栈配合方便。最终界面中的饼图展示分类支出占比，条形图展示各分类支出金额的比较。

当前月份的账单、总额和图表应使用一致的数据范围。课程后面的运行截图展示了这三者的对应结果；库选好和代码编译通过，只完成了实现过程的一部分。

## 15. 选择导出库：SheetJS

统计图表编译通过后，Claude 提出 Excel 导出方案：

| 方案 | 本项目中的取舍 |
|---|---|
| A：SheetJS（xlsx） | 支持导出 `.xlsx`、`.csv`，适合本课以账单数据为主的导出 |
| B：ExcelJS | 适合需要进一步控制字体、颜色、列宽、合并单元格等格式的场景 |

<img src="images/04-10-export-options.png" alt="课程中的 SheetJS 与 ExcelJS 方案比较" width="760">

课程选 **A：SheetJS**，需求重点是把日期、分类、金额和备注导出成表格，未要求复杂的报表样式。

截图中关于包体大小、流行程度以及样式限制的措辞，是当时 AI 的方案说明。本课只据此记录选择理由，不把“几乎所有网站都用它”等表述作为事实结论。

完成后应验证两种格式都能读取，字段和记录与应用内数据对应；不能只检查文件扩展名或下载是否完成。

## 16. 保留一次真实的导入修复

导出实现后，课程展示了 `export.ts` 的一次修改。原来的导入：

```typescript
import { getRecordsByMonth } from "../data/db";
```

修改为：

```typescript
import { getRecordsByMonth, db } from "../data/db";
```

同时删除函数中的动态导入：

```typescript
const { db } = await import("../data/db");
```

后续使用的查询片段仍为：

```typescript
const allRecords = await db.records.orderBy("date").reverse().toArray();
```

<img src="images/04-11-export-import-fix.png" alt="课程 export.ts 的数据库导入修改" width="760">

这次修改把 `db` 改为文件顶部直接导入，避免在函数内再次动态获取。讲义没有展示对应的完整错误日志，因此不能进一步断言原写法必然导致某一种运行异常。

本课称其为“Bug 自修复”，应理解为 AI 阅读文件、修改实现并继续检查的过程；具体是否修好，仍要看构建结果和实际导出行为。

## 17. 生成图标，编译 Rust 并修复 Tauri 配置

前端完成后进入桌面打包。课程先让 AI 准备 `app-icon.svg`，再在**项目根目录**执行：

```bash
npx tauri icon app-icon.svg
```

它生成 Tauri 使用的各平台图标，不是在生成安装包。随后在 **`src-tauri` 目录**执行：

```bash
cargo build
```

macOS 演示中先用 `source "$HOME/.cargo/env"` 加载环境。首次 Rust 编译会获取依赖，不能因为耗时较长就直接认为失败。

### 配置错误及实际修复

课程中第一次 Rust 编译失败。Claude 阅读 `tauri.conf.json`，删除了误放在 `app` 对象下的这一行：

```json
"title": "黑马记账",
```

<img src="images/04-12-tauri-config-fix.png" alt="课程删除 app 下错误的 title 字段并重新构建" width="760">

窗口标题仍应保留在 `app.windows` 中的窗口配置里。删除的是**错误层级的字段**，不是把所有窗口标题都删除。

修改后再次在 `src-tauri` 运行 `cargo build`，讲义记录 Rust 编译成功。这里是 Rust 构建检查；正式发行打包仍在下一步执行。

## 18. 打包 macOS 应用

Rust 编译通过后，回到**项目根目录**执行：

```bash
npx tauri build
```

该过程会按 `tauri.conf.json` 的构建配置执行前端构建、Rust 发行构建和所选格式的打包。课程的 `beforeBuildCommand` 是 `npm run build`，所以这一阶段也会运行 TypeScript 检查和 Vite 构建。

讲义演示在 Apple Silicon Mac 上生成 macOS 应用与 DMG，并展示安装窗口：

<img src="images/04-13-macos-dmg.png" alt="课程生成的黑马记账 macOS DMG 安装窗口" width="760">

后续说明中的 DMG 名称是 `黑马记账_1.0.0_aarch64.dmg`。`aarch64` 对应该次构建的处理器架构，不能据此认为已经同时产出了 Intel Mac 版本。

打开 DMG 后，将应用拖入 `Applications`，再运行“黑马记账”，检查记账、统计和导出界面。讲义还展示了 macOS 运行效果和导出的表格。

**补充：**DMG 是分发用的磁盘映像，里面的 `.app` 才是应用。该次打包成功不等于 Windows 版本也已生成；课程接下来切换到 Windows 继续构建。

## 19. 将项目复制到 Windows，继续同一个打包任务

讲义主要开发过程在 macOS 进行。生成 Windows 程序时，将项目文件夹复制或同步到 Windows，用 VSCode 打开该项目。

课程原始续接提示词：

```text
请帮我将项目打包为Windows的exe程序；
此项目内容在MacOS系统下构建，已经完成了Mac版本的程序打包，所以复制到Windows系统下，你帮我继续打包Windows版本的程序exe
```

这段说明保留了已有进度与当前目标，让 AI 继续构建 Windows 版本，而不是重写整个记账产品。

Claude 先检查 Node.js、npm、Rust、Cargo，发现 Windows 缺少 Rust。课程使用 `winget` 安装 rustup；截图可见的核心命令为：

```powershell
winget install --id Rustlang.Rustup
```

这里省略课程截图中未完整显示的其他参数，不补造完整安装命令。安装后，重新让当前终端识别 Cargo 路径，再验证 `rustc --version` 和 `cargo --version`。

**补充：**课程 Windows 截图里的部分命令仍通过 Bash 执行，所以出现 `export PATH=...`、`2>/dev/null`。PowerShell 中可以在安装后重新打开终端再检查版本，不要直接照搬 Bash 的 `export` 写法。

项目源码可以跨平台继续使用，但 macOS 的 Rust 安装、编译缓存和部分 `node_modules` 内容不能直接当成 Windows 环境。

## 20. Windows 编译方案：选择 MSVC

Rust 安装成功后，课程发现仍缺少微软 C++ 编译工具与链接器。Claude 按项目约定列出两种选择：

| 方案 | 课程讨论的取舍 | 结果 |
|---|---|---|
| A：MSVC + Visual Studio Build Tools | Windows 原生工具链，Tauri 常用方案；下载和安装体积较大 | 选 A |
| B：切换 GNU 工具链 | 下载可能较小，但需处理相关依赖的兼容性 | 未选 |

<img src="images/04-14-windows-compiler-options.png" alt="课程的 MSVC 与 GNU 编译方案比较" width="680">

课程选择 **Visual Studio Build Tools**，安装过程中会弹出 Visual Studio Installer。完成后，Claude 检查 MSVC 目录及 `cl.exe` 等工具。

**补充：**VSCode 是编辑器，Visual Studio Build Tools 提供 Windows 的编译与链接工具，安装 VSCode 不等于已经具备 MSVC。复现时需要 C++ 桌面开发所需组件及 Windows SDK；这些是构建 Tauri 的平台依赖。

截图中的“约 2–3 GB”“能确保一次性成功”是当时 AI 的描述。实际下载量与成功与否还取决于组件、网络和项目状态。

## 21. 清理旧产物、重装依赖并在 Windows 构建

课程在 Windows 工具链就绪后进行三类处理：

1. 在 `src-tauri` 执行 `cargo clean`，清理从 macOS 带来的 Rust 构建产物。
2. 重装 npm 依赖，处理课程中遇到的 macOS → Windows 符号链接兼容问题。
3. 重新构建前端，再执行 Tauri 的 Windows 发行构建。

对应核心命令如下。先在 **`src-tauri` 目录**：

```powershell
cargo clean
```

随后回到包含 `package.json` 的**项目根目录**：

```powershell
npm install
npm run build
npx tauri build
```

这组命令按课程处理顺序整理，省略了作者路径和 Bash 环境包装。`cargo clean` 删除的是 Rust 构建目录里的产物与缓存，不是业务源代码；课程记录清理了约 `3.6 GB`，该体积不具有普遍性。

`npm install` 列出了依赖重新安装环节，讲义没有完整展开相关目录清理命令，本篇不补写递归删除操作。

构建时区分三个结果：

| 阶段 | 能说明什么 |
|---|---|
| 前端构建完成 | 前端资源可以生成 |
| Rust 发行构建完成 | Windows 原生可执行程序已经编译 |
| 安装包生成完成 | 指定的 MSI/NSIS 等分发格式打包成功 |

后一个阶段失败时，前一个阶段的 EXE 可能已经存在；本课接下来正好遇到这一情况。

## 22. WiX 下载超时，改用 NSIS 后核对实际产物

Windows 构建中，课程记录 **EXE 已生成，约 `8.8 MB`**，但下载 WiX 时访问 GitHub 超时，导致 **MSI 安装包未生成**。

<img src="images/04-15-wix-download-timeout.png" alt="课程中 EXE 已生成但 WiX 下载超时的记录" width="680">

Claude 先检查 WiX 缓存位置，再尝试解决下载问题。随后改为编辑 `src-tauri/tauri.conf.json`，将安装包策略切换为 NSIS，重新执行：

```powershell
npx tauri build
```

<img src="images/04-16-nsis-retry.png" alt="课程修改 Tauri 配置后尝试以 NSIS 格式重新打包" width="680">

讲义把这一段叫“自动切换网络方案”，截图实际展示的是**切换打包工具和格式**，没有展示代理设置。本篇也不补写代理配置。

**实际可确认的最终交付**是：项目根目录的 `黑马记账.exe`，约 `8.8 MB`；原始程序保留在 `src-tauri/target/release/heimajizhang.exe`。

<img src="images/04-17-windows-output.png" alt="课程列出的 Windows EXE 输出路径与运行要求" width="620">

最终总结与运行截图证明了可运行 EXE 的交付，讲义没有列出独立 NSIS 安装器的文件名和路径，因此不能进一步写成“MSI 与 NSIS 安装器都已生成”。

**补充：**`.exe` 既可以是应用本身，也可以是安装器，须结合路径和行为判断。本课展示的是无需安装、双击运行的应用 EXE。Tauri 依赖系统 WebView2，目标 Windows 仍需具备相应运行时；“无需安装”不等于完全没有运行环境要求。

## 23. 运行应用，核对记账、统计与导出

讲义在 Windows 双击 `黑马记账.exe`，展示了三个页面入口：**记账、统计、导出**。

### 记账界面

页面包含月份、当月支出、金额、分类、日期和备注。课程设计的一级分类包括餐饮饮食、交通出行、购物消费、住房居住、医疗健康、休闲娱乐、人情往来、教育学习、金融保险、其他支出；需求还包括二级分类。

<img src="images/04-18-windows-record.png" alt="课程中的 Windows 记账页面" width="440">

### 用同一组账单核对统计

课程演示的三笔 Windows 账单为：

| 日期 | 分类 | 金额（元） | 备注 |
|---|---|---:|---|
| 2026-06-18 | 房租 | 6646 | 空 |
| 2026-06-18 | 飞机 | 1110 | 空 |
| 2026-06-18 | 午餐 | 111 | 空 |

总额为 `6646 + 1110 + 111 = 7867` 元，统计页显示 `￥7867.00`。房租归住房居住、飞机归交通出行、午餐归餐饮饮食，图表据此显示分类金额与占比。

<img src="images/04-19-windows-statistics.png" alt="课程中的月度支出占比与对比图，总额7867元" width="440">

### 对照导出文件

Excel 截图中的列名是“日期、分类、金额、备注”，三条记录与上述账单一致：

<img src="images/04-20-excel-output.png" alt="课程导出的 Excel 账单数据" width="900">

CSV 截图中的文件名是 `黑马记账_2026年6月.csv`，内容为：

```csv
日期,分类,金额,备注
2026-06-18,房租,6646,
2026-06-18,飞机,1110,
2026-06-18,午餐,111,
```

末尾的逗号表示备注为空，仍保留第四个字段。

<img src="images/04-21-csv-output.png" alt="课程导出的 CSV 原始内容" width="520">

**补充：复现时的核对方法。**按上述数据录入后，检查总额、分类图表和两种导出结果是否一致；关闭并重新打开应用，确认记录仍在；切换月份，检查统计和导出范围。后两项是建议补做的验证，讲义截图没有证明已完成。

本篇只整理课程中的运行结果，没有在当前电脑安装依赖、构建或运行这款记账应用。

## 24. 课程中的消耗费用参考

讲义最后注明：该次操作全程使用 **DeepSeek V4 Pro**，消耗约 **人民币 `1.3` 元**。

这是作者当次开发的费用记录，受模型、计费方式、上下文长度和重试次数影响，不能据此推算每次复现都会花相同金额。Claude Code 是本课使用的开发工具，具体费用还与它所连接的模型服务有关。

本课最终展示了从产品需求到 macOS 应用、Windows EXE，以及记账、统计和导出的完整开发过程。